In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/synthetic/payment_events.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,customer_id,transaction_id,transaction_amount,payment_method,payment_status,failure_reason,attempt_number,previous_successful_payments,previous_failed_payments,customer_tenure_days,recovery_attempted,recovery_action,recovered,recovered_amount
0,CUST_00447,TXN_000001,13.75,card,failed,insufficient_funds,1,6,1,365,True,send_reminder,False,0.00
1,CUST_03870,TXN_000002,21.48,netbanking,failed,network_error,1,6,1,331,True,send_reminder,True,21.48
2,CUST_03273,TXN_000003,41.17,card,success,NaN,2,10,2,305,False,NaN,False,0.00
3,CUST_02194,TXN_000004,20.03,netbanking,failed,insufficient_funds,1,7,0,859,True,retry_payment,False,0.00
4,CUST_02165,TXN_000005,26.50,card,success,NaN,1,9,1,1225,False,NaN,False,0.00


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customer_id                   10000 non-null  str    
 1   transaction_id                10000 non-null  str    
 2   transaction_amount            10000 non-null  float64
 3   payment_method                10000 non-null  str    
 4   payment_status                10000 non-null  str    
 5   failure_reason                1479 non-null   str    
 6   attempt_number                10000 non-null  int64  
 7   previous_successful_payments  10000 non-null  int64  
 8   previous_failed_payments      10000 non-null  int64  
 9   customer_tenure_days          10000 non-null  int64  
 10  recovery_attempted            10000 non-null  bool   
 11  recovery_action               1000 non-null   str    
 12  recovered                     10000 non-null  bool   
 13  recovered_amo

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_amount,10000.0,45.769185,44.403092,0.99,19.3775,32.585,56.4725,916.90
attempt_number,10000.0,2.009200,0.820843,1.00,1.0000,2.000,3.0000,3.00
previous_successful_payments,10000.0,8.006000,2.849101,0.00,6.0000,8.000,10.0000,21.00
previous_failed_payments,10000.0,1.511300,1.238235,0.00,1.0000,1.000,2.0000,8.00
customer_tenure_days,10000.0,748.488900,432.334771,1.00,374.0000,752.000,1122.0000,1499.00
recovered_amount,10000.0,2.759657,15.493920,0.00,0.0000,0.000,0.0000,400.76


In [5]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values(
    "missing_pct",
    ascending=False,
)

,missing_count,missing_pct
recovery_action,9000,90.00
failure_reason,8521,85.21
transaction_id,0,0.00
transaction_amount,0,0.00
payment_method,0,0.00
customer_id,0,0.00
payment_status,0,0.00
attempt_number,0,0.00
previous_failed_payments,0,0.00
previous_successful_payments,0,0.00


In [6]:
funnel = {
    "Total transactions": len(df),
    "Failed payments": (df["payment_status"] == "failed").sum(),
    "Recovery attempts": df["recovery_attempted"].sum(),
    "Recovered payments": df["recovered"].sum(),
}

funnel

{'Total transactions': 10000,
 'Failed payments': np.int64(1479),
 'Recovery attempts': np.int64(1000),
 'Recovered payments': np.int64(594)}

In [7]:
failed = (df["payment_status"] == "failed").sum()
attempted = df["recovery_attempted"].sum()
recovered = df["recovered"].sum()

print(f"Failure rate: {failed / len(df):.2%}")
print(f"Recovery attempt rate among failures: {attempted / failed:.2%}")
print(f"Recovery success rate among attempts: {recovered / attempted:.2%}")
print(f"Overall recovered transaction rate: {recovered / len(df):.2%}")

Failure rate: 14.79%
Recovery attempt rate among failures: 67.61%
Recovery success rate among attempts: 59.40%
Overall recovered transaction rate: 5.94%


In [8]:
failure_analysis = (
    df[df["recovery_attempted"]]
    .groupby("failure_reason")
    .agg(
        attempts=("recovery_attempted", "size"),
        recovered=("recovered", "sum"),
        recovery_rate=("recovered", "mean"),
    )
    .sort_values("recovery_rate", ascending=False)
)

failure_analysis

,attempts,recovered,recovery_rate
failure_reason,,,
network_error,211,137,0.649289
expired_card,197,126,0.639594
card_declined,204,126,0.617647
authentication_failed,187,115,0.614973
insufficient_funds,201,90,0.447761


In [9]:
failure_analysis["recovery_rate"] = (
    failure_analysis["recovery_rate"] * 100
).round(2)

failure_analysis

,attempts,recovered,recovery_rate
failure_reason,,,
network_error,211,137,64.93
expired_card,197,126,63.96
card_declined,204,126,61.76
authentication_failed,187,115,61.50
insufficient_funds,201,90,44.78


In [10]:
recovery_df = df[df["recovery_attempted"]].copy()

recovery_df["successful_payment_bucket"] = pd.cut(
    recovery_df["previous_successful_payments"],
    bins=[-1, 3, 6, 9, 12, np.inf],
    labels=[
        "0-3",
        "4-6",
        "7-9",
        "10-12",
        "13+",
    ],
)

success_history = (
    recovery_df
    .groupby("successful_payment_bucket", observed=True)
    .agg(
        attempts=("recovered", "size"),
        recovered=("recovered", "sum"),
        recovery_rate=("recovered", "mean"),
    )
)

success_history["recovery_rate"] *= 100
success_history

,attempts,recovered,recovery_rate
successful_payment_bucket,,,
0-3,31,18,58.064516
4-6,275,151,54.909091
7-9,402,234,58.208955
10-12,227,147,64.757709
13+,65,44,67.692308


In [11]:
recovery_df["failed_payment_bucket"] = pd.cut(
    recovery_df["previous_failed_payments"],
    bins=[-1, 1, 2, 3, np.inf],
    labels=[
        "0-1",
        "2",
        "3",
        "4+",
    ],
)

failure_history = (
    recovery_df
    .groupby("failed_payment_bucket", observed=True)
    .agg(
        attempts=("recovered", "size"),
        recovered=("recovered", "sum"),
        recovery_rate=("recovered", "mean"),
    )
)

failure_history["recovery_rate"] *= 100
failure_history

,attempts,recovered,recovery_rate
failed_payment_bucket,,,
0-1,551,340,61.705989
2,264,152,57.575758
3,124,69,55.645161
4+,61,33,54.098361


In [12]:
recovery_df["amount_bucket"] = pd.qcut(
    recovery_df["transaction_amount"],
    q=4,
    duplicates="drop",
)

amount_analysis = (
    recovery_df
    .groupby("amount_bucket", observed=True)
    .agg(
        attempts=("recovered", "size"),
        recovered=("recovered", "sum"),
        recovery_rate=("recovered", "mean"),
        average_amount=("transaction_amount", "mean"),
    )
)

amount_analysis["recovery_rate"] *= 100
amount_analysis

,attempts,recovered,recovery_rate,average_amount
amount_bucket,,,,
"(2.509, 19.552]",250,153,61.2,13.05892
"(19.552, 32.13]",250,149,59.6,25.41732
"(32.13, 55.382]",250,133,53.2,43.28424
"(55.382, 400.76]",250,159,63.6,101.32952


In [13]:
recovery_df["tenure_bucket"] = pd.cut(
    recovery_df["customer_tenure_days"],
    bins=[0, 180, 365, 730, 1095, np.inf],
    labels=[
        "0-180 days",
        "181-365 days",
        "366-730 days",
        "731-1095 days",
        "1096+ days",
    ],
)

tenure_analysis = (
    recovery_df
    .groupby("tenure_bucket", observed=True)
    .agg(
        attempts=("recovered", "size"),
        recovered=("recovered", "sum"),
        recovery_rate=("recovered", "mean"),
    )
)

tenure_analysis["recovery_rate"] *= 100
tenure_analysis

,attempts,recovered,recovery_rate
tenure_bucket,,,
0-180 days,126,64,50.793651
181-365 days,118,70,59.322034
366-730 days,229,123,53.711790
731-1095 days,234,158,67.521368
1096+ days,293,179,61.092150


In [14]:
TARGET = "recovered"

model_df = df[df["recovery_attempted"]].copy()

print("Modeling rows:", len(model_df))
print("Target distribution:")
print(model_df[TARGET].value_counts())
print()
print("Target rate:")
print(model_df[TARGET].mean())

Modeling rows: 1000
Target distribution:
recovered
True     594
False    406
Name: count, dtype: int64

Target rate:
0.594


In [15]:
potential_leakage = [
    "recovered",
    "recovered_amount",
]

print("Potential leakage columns:")
for column in potential_leakage:
    print("-", column)

Potential leakage columns:
- recovered
- recovered_amount


In [16]:
candidate_features = [
    "transaction_amount",
    "payment_method",
    "failure_reason",
    "attempt_number",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_tenure_days",
]

print("Candidate features:")
for feature in candidate_features:
    print("-", feature)

Candidate features:
- transaction_amount
- payment_method
- failure_reason
- attempt_number
- previous_successful_payments
- previous_failed_payments
- customer_tenure_days


In [17]:
model_df[candidate_features + [TARGET]].head()

,transaction_amount,payment_method,failure_reason,attempt_number,previous_successful_payments,previous_failed_payments,customer_tenure_days,recovered
0,13.75,card,insufficient_funds,1,6,1,365,False
1,21.48,netbanking,network_error,1,6,1,331,True
3,20.03,netbanking,insufficient_funds,1,7,0,859,False
14,17.00,card,authentication_failed,1,9,1,103,True
17,54.15,card,card_declined,3,7,2,745,True


In [18]:
# ============================================================
# EDA CONCLUSIONS
# ============================================================

# Dataset
# - 10,000 synthetic payment events
# - 14 columns
# - 1,479 failed payments
# - 1,000 recovery attempts
# - 594 successful recoveries

# Recovery Funnel
# - Payment failure rate: 14.79%
# - Recovery attempt rate among failed payments: 67.61%
# - Recovery success rate among recovery attempts: 59.40%

# Important Observations
# - Failure reason may be predictive of recovery success.
# - Previous successful and failed payment history contains behavioral signal.
# - Customer tenure and transaction amount are potential predictive features.
# - failure_reason and recovery_action contain expected structural missing values.
# - recovered is the prediction target.
# - recovered_amount is excluded because it is known after the recovery outcome.
# - customer_id and transaction_id are identifiers, not predictive features.

# Modeling Population
# The first recovery prediction model will use transactions where:
# recovery_attempted == True
#
# Target:
# recovered